<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session02_vibe_coding_workflow_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۲: وایب کدینگ: برنامه‌نویسی با دستیار هوش مصنوعی

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session02_vibe_coding_workflow.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install openai
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

In [ ]:
import re
import subprocess

SYSTEM = (
    "You are a senior Python developer. Write clean, readable code with docstrings. "
    "Return the complete content of the requested module inside one ```python code block and nothing else."
)


def extract_code(reply):
    match = re.search(r"```(?:python)?\s*\n(.*?)```", reply, re.S)
    return match.group(1) if match else reply


def run_tests(test_file):
    result = subprocess.run(["python", "-m", "pytest", "-q", test_file], capture_output=True, text=True)
    return result.returncode == 0, result.stdout[-2500:]


def coding_loop(spec, tests, module="solution.py", test_file="test_solution.py", max_attempts=3, plan=None):
    with open(test_file, "w", encoding="utf-8") as f:
        f.write(tests)
    request = f"{spec}\n\nThe code must pass these tests:\n\n{tests}"
    if plan:
        request += f"\n\nFollow this plan:\n{plan}"
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": request}]
    for attempt in range(1, max_attempts + 1):
        reply = chat(client, MODEL, messages=messages, temperature=0.2)
        with open(module, "w", encoding="utf-8") as f:
            f.write(extract_code(reply))
        passed, report = run_tests(test_file)
        print(f"attempt {attempt}: {report.strip().splitlines()[-1]}")
        if passed:
            return True
        messages += [{"role": "assistant", "content": reply},
                     {"role": "user", "content": f"The tests failed:\n\n{report}\n\nFix the code and return the complete file."}]
    return False

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

دو مورد آزمایشی جدید به `test_solution.py` اضافه کنید: «۱٫۲۵۰٫۰۰۰ تومان» و ورودی خالی. مشخصات را هم به‌روز کنید و حلقه `coding_loop` را دوباره اجرا کنید.

</div>

In [ ]:
SPEC_V2 = """# Spec: parse_price (version 2)

Write a Python module `solution.py` with one function `parse_price(text: str) -> int`
that converts a Persian price string to an integer amount in toman.

- Digits may be Persian (۰-۹), Arabic-Indic (٠-٩) or English.
- "٫" is a decimal separator. "," and "٬" are thousands separators.
- "." is a decimal separator, EXCEPT when the number has the form 1.250.000
  (groups of exactly three digits): then "." is a thousands separator.
- Units: "تومان" (x1), "هزار تومان" (x1,000), "میلیون تومان" (x1,000,000), "میلیارد تومان" (x1,000,000,000).
- "رایگان" means 0.
- An empty string, or a string without a number, raises ValueError.
- Use only the standard library.
"""

TESTS_V2 = """import pytest
from solution import parse_price


@pytest.mark.parametrize("text, expected", [
    ("45000 تومان", 45_000),
    ("۴۵,۰۰۰ تومان", 45_000),
    ("۲۵۰ هزار تومان", 250_000),
    ("1.5 میلیون تومان", 1_500_000),
    ("۱۲٫۵ میلیون تومان", 12_500_000),
    ("3 میلیارد تومان", 3_000_000_000),
    ("۱٫۲۵۰٫۰۰۰ تومان", 1_250_000),
    ("1.250.000 تومان", 1_250_000),
    ("رایگان", 0),
])
def test_parse_price(text, expected):
    assert parse_price(text) == expected


@pytest.mark.parametrize("text", ["", "   ", "قیمت ندارد"])
def test_invalid(text):
    with pytest.raises(ValueError):
        parse_price(text)
"""

print("passed" if coding_loop(SPEC_V2, TESTS_V2) else "failed")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

ورودی «۱٫۲۵۰٫۰۰۰ تومان» ابهام واقعی دارد: «٫» در فارسی ممیز است، ولی اینجا کاربر آن را جداکنندهٔ هزارگان به کار برده است. مشخصات باید این ابهام را **صریحاً** حل کند. ما قاعده گذاشتیم که گروه‌های سه‌رقمی پشت‌سرهم یعنی جداکنندهٔ هزارگان. دقت کنید این قاعده را برای هر دو جداکننده، یعنی `.` و `٫`، نوشته‌ایم و تست هر دو حالت را داریم.

این تمرین درس اصلی جلسه را نشان می‌دهد: وقتی ورودی جدیدی کشف می‌شود، اول **مشخصات و تست** به‌روز می‌شوند و بعد کد. اگر مدل محلی بعد از سه تلاش موفق نشد، گزارش آخرین شکست را بخوانید: معمولاً یکی از موارد مرزی است و می‌توانید با یک جملهٔ دقیق‌تر در مشخصات به مدل کمک کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

حلقه را طوری تغییر دهید که پیش از نوشتن کد، از مدل یک برنامه پنج‌خطی بخواهد و آن را چاپ کند. آیا کیفیت کد تغییر می‌کند؟

</div>

In [ ]:
PLAN_PROMPT = ("Before writing any code, write a numbered plan of at most five short lines for "
               "implementing this specification. Do not write code.\n\n")

plan = chat(client, MODEL, PLAN_PROMPT + SPEC_V2, temperature=0)
print("PLAN:\n" + plan + "\n")
print("passed" if coding_loop(SPEC_V2, TESTS_V2, plan=plan) else "failed")
print(open("solution.py", encoding="utf-8").read())

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

برنامه را **قبل** از تولید کد بخوانید. اگر قدمی جا افتاده، مثلاً تبدیل ارقام فارسی یا حالت رشتهٔ خالی، همین‌جا اصلاحش کنید و برنامهٔ اصلاح‌شده را به `coding_loop` بدهید. این کار خیلی ارزان‌تر از اصلاح کد است.

اثر روی کیفیت کد از یک اجرا قابل نتیجه‌گیری نیست. برای سنجش منصفانه، هر حالت را چند بار، مثلاً ۵ بار، اجرا کنید و تعداد دفعات موفقیت در تلاش اول را بشمارید. در عمل، برنامه‌ریزی برای کارهای چندمرحله‌ای بیشترین فایده را دارد. برای تابع‌های کوچک ممکن است تفاوتی دیده نشود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

برای تابع `normalize_mobile(text)` که شماره موبایل ایرانی را به قالب `09xxxxxxxxx` تبدیل می‌کند، مشخصات و تست بنویسید و با `coding_loop` پیاده‌سازی کنید. ورودی‌هایی مثل `+98 912 345 6789` و `۰۹۱۲-۳۴۵-۶۷۸۹` را پوشش دهید.

</div>

In [ ]:
MOBILE_SPEC = """# Spec: normalize_mobile

Write a Python module `mobile.py` with one function `normalize_mobile(text: str) -> str`
that returns an Iranian mobile number in the form 09xxxxxxxxx (11 English digits).

- Accept Persian, Arabic-Indic and English digits, spaces, dashes and parentheses.
- Accept the prefixes +98, 0098, 98 and a missing leading zero (9123456789).
- Raise ValueError for landline numbers (not starting with 09), numbers of the wrong length,
  and strings without digits.
- Use only the standard library.
"""

MOBILE_TESTS = """import pytest
from mobile import normalize_mobile


@pytest.mark.parametrize("raw, expected", [
    ("09123456789", "09123456789"),
    ("0912 345 6789", "09123456789"),
    ("+98 912 345 6789", "09123456789"),
    ("00989123456789", "09123456789"),
    ("9123456789", "09123456789"),
    ("۰۹۱۲-۳۴۵-۶۷۸۹", "09123456789"),
    ("(0912) 345-6789", "09123456789"),
])
def test_valid(raw, expected):
    assert normalize_mobile(raw) == expected


@pytest.mark.parametrize("raw", ["02112345678", "0912345678", "091234567890", "", "شماره ندارم"])
def test_invalid(raw):
    with pytest.raises(ValueError):
        normalize_mobile(raw)
"""

ok = coding_loop(MOBILE_SPEC, MOBILE_TESTS, module="mobile.py", test_file="test_mobile.py")
print("passed" if ok else "failed")
print(open("mobile.py", encoding="utf-8").read())

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

نکته‌هایی که تست‌ها باید پوشش دهند و مدل‌ها زیاد اشتباه می‌کنند:

- در پایتون ۳، `\d` و `\D` ارقام یونیکد، یعنی فارسی و عربی، را هم «رقم» حساب می‌کنند. پس `re.sub(r"\D", "", text)` ارقام فارسی را حذف نمی‌کند ولی به انگلیسی هم تبدیلشان نمی‌کند. تبدیل باید جداگانه، مثلاً با `str.translate`، انجام شود.
- پیشوند `98` فقط وقتی باید حذف شود که بعد از آن دقیقاً ۱۰ رقم بیاید. در غیر این صورت شماره‌ای مثل `0912...98...` خراب می‌شود.
- شمارهٔ ثابت تهران (`021...`) هم ۱۱ رقم است. شرط «۱۱ رقم» کافی نیست و باید با `09` شروع شود.

یک پیاده‌سازی مرجع از این تابع در جلسهٔ ۳ آمده است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

Continue و Ollama را روی سیستم خودتان راه بیندازید و با کمک آن یک اسکریپت بنویسید که فایل `products.csv` نوین‌کالا را بخواند و ارزان‌ترین کالای هر دسته را چاپ کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

این تمرین روی سیستم خودتان انجام می‌شود. مراحل:

1. Ollama را نصب کنید و مدل‌ها را بگیرید: `ollama pull qwen2.5-coder:7b` و `ollama pull qwen2.5-coder:1.5b`.
2. افزونهٔ **Continue** را در VS Code نصب کنید و در تنظیمات آن، مدل چت را `qwen2.5-coder:7b` و مدل تکمیل خودکار را `qwen2.5-coder:1.5b` با فراهم‌کنندهٔ Ollama انتخاب کنید.
3. یک فایل خالی `cheapest.py` بسازید، با `Ctrl+L` پنل چت را باز کنید و درخواستی دقیق بنویسید، مثلاً: «اسکریپتی بنویس که `data/products.csv` را با pandas بخواند و برای هر مقدار ستون `category` ارزان‌ترین کالا را بر اساس `price_toman` با نام و قیمت چاپ کند. کالاهای با `stock` صفر را کنار بگذار.»
4. کد پیشنهادی را بخوانید، اجرا کنید و با داده مقایسه کنید.

یک پاسخ درست، که می‌توانید خروجی ابزار را با آن مقایسه کنید:

</div>

In [ ]:
import pandas as pd

products = pd.read_csv("https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/data/products.csv")
in_stock = products[products.stock > 0]
cheapest = in_stock.loc[in_stock.groupby("category").price_toman.idxmin(),
                        ["category", "name", "price_toman"]]
for row in cheapest.itertuples():
    print(f"{row.category:20} {row.price_toman:>12,} toman  {row.name}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

خطای رایج ابزارها در این کار، فراموش کردن کالاهای ناموجود است؛ کالایی با موجودی صفر ارزان‌ترین گزینهٔ دسته نیست. خطای رایج دیگر استفاده از `sort_values().groupby().first()` است که درست کار می‌کند ولی نیت کد را کمتر روشن می‌کند. اگر در درخواست خود دربارهٔ موجودی چیزی نگفته بودید، این نمونهٔ خوبی است از اینکه **مشخصات ناقص به کد ناقص می‌رسد**.

</div>